In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

try:
    import xlsxwriter                      
    EXCEL_ENGINE = "xlsxwriter"
except ImportError:
    EXCEL_ENGINE = "openpyxl"             

BASE_DIR   = Path(r"C:\Users\LAP TECH\Desktop\DEPI final Project")
FILE_PATH  = BASE_DIR / "DescriptionDataCoSupplyChain.csv"
OUTPUT_DIR = BASE_DIR / "files"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)   

desc = pd.read_csv(FILE_PATH)
desc.columns = ["original_field", "description"]
print("Shape before cleaning:", desc.shape)
desc.head(4)

Shape before cleaning: (52, 2)


,original_field,description
0,Type,: Type of transaction made
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased product
3,Benefit per order,: Earnings per order placed


In [2]:
desc["original_field"] = desc["original_field"].str.strip()

def clean_description(text):
    text = text.strip()
    text = re.sub(r"^:\s*", "", text)       
    text = re.sub(r"\s*:\s*", ": ", text)    
    text = re.sub(r"\s*,\s*", ", ", text)   
    text = re.sub(r"\s+", " ", text)         
    return text.strip()

desc["description"] = desc["description"].map(clean_description)
desc[["original_field", "description"]].iloc[[0, 1, 5, 40]]

,original_field,description
0,Type,Type of transaction made
1,Days for shipping (real),Actual shipping days of the purchased product
5,Delivery Status,"Delivery status of orders: Advance shipping, Late delivery, Shipping canceled, Shippin..."
40,Order Region,"Region of the world where the order is delivered: Southeast Asia, South Asia, Oceania,..."


In [3]:
is_region = desc["original_field"] == "Order Region"

fixed = (desc.loc[is_region, "description"]
         .str.replace("Northern, ", "Northern Europe, ", regex=False)
         .str.replace(", Europe, ", ", ", regex=False))
desc.loc[is_region, "description"] = fixed

region_list = desc.loc[is_region, "description"].iloc[0].split(": ", 1)[1].split(", ")
print(len(region_list), "regions")
desc.loc[is_region, "description"].iloc[0]

23 regions


'Region of the world where the order is delivered: Southeast Asia, South Asia, Oceania, Eastern Asia, West Asia, West of USA, US Center, West Africa, Central Africa, North Africa, Western Europe, Northern Europe, Caribbean, South America, East Africa, Southern Europe, East of USA, Canada, Southern Africa, Central Asia, Central America, Eastern Europe, South of USA'

In [4]:
def to_snake(name):
    name = name.strip().lower()
    name = re.sub(r"[^0-9a-z]+", "_", name)
    return name.strip("_")

desc["column_name"] = desc["original_field"].map(to_snake)
desc["column_name"] = desc["column_name"].replace({
    "type": "payment_type",
    "order_date_dateorders": "order_date",
    "shipping_date_dateorders": "shipping_date",
})
desc["source"] = "original dictionary"

assert desc["column_name"].is_unique, 
desc[["original_field", "column_name"]].head(8)

,original_field,column_name
0,Type,payment_type
1,Days for shipping (real),days_for_shipping_real
2,Days for shipment (scheduled),days_for_shipment_scheduled
3,Benefit per order,benefit_per_order
4,Sales per customer,sales_per_customer
5,Delivery Status,delivery_status
6,Late_delivery_risk,late_delivery_risk
7,Category Id,category_id


In [5]:
dropped = {
    "benefit_per_order":        "Dropped: identical to order_profit_per_order",
    "sales_per_customer":       "Dropped: identical to order_item_total",
    "order_item_cardprod_id":   "Dropped: identical to product_card_id",
    "product_category_id":      "Dropped: identical to category_id",
    "order_item_product_price": "Dropped: identical to product_price",
    "order_customer_id":        "Dropped: identical to customer_id",
    "customer_email":           "Dropped: masked, no useful information",
    "customer_password":        "Dropped: masked, no useful information",
    "product_description":      "Dropped: empty column",
    "product_status":           "Dropped: constant value (0 = available for every row)",
    "product_image":            "Dropped: image link only",
}

desc["in_cleaned_dataset"] = ~desc["column_name"].isin(dropped)
desc["cleaning_note"] = desc["column_name"].map(dropped).fillna("")

extra = pd.DataFrame([
    ("Order Zipcode", "order_zipcode", "Not described in the original dictionary", False,
     "Dropped: about 86% missing", "missing from original dictionary"),
    ("(added)", "order_year",       "Year the order was made (from order_date)", True, "Added during cleaning", "added in cleaning"),
    ("(added)", "order_month",      "Month number of the order (1-12)", True, "Added during cleaning", "added in cleaning"),
    ("(added)", "order_month_name", "Month name of the order", True, "Added during cleaning", "added in cleaning"),
    ("(added)", "order_day_name",   "Day of the week the order was made", True, "Added during cleaning", "added in cleaning"),
    ("(added)", "shipping_delay_days",
     "Actual shipping days minus scheduled days (positive = late, negative = early)", True,
     "Added during cleaning", "added in cleaning"),
], columns=["original_field", "column_name", "description", "in_cleaned_dataset", "cleaning_note", "source"])

desc = pd.concat([desc, extra], ignore_index=True)
desc = desc[["original_field", "column_name", "description", "in_cleaned_dataset", "cleaning_note", "source"]]
desc["in_cleaned_dataset"] = desc["in_cleaned_dataset"].map({True: "Yes", False: "No"})
desc.tail(8)

,original_field,column_name,description,in_cleaned_dataset,cleaning_note,source
50,Shipping date (DateOrders),shipping_date,Exact date and time of shipment,Yes,,original dictionary
51,Shipping Mode,shipping_mode,"The following shipping modes are presented: Standard Class, First Class, Second Class,...",Yes,,original dictionary
52,Order Zipcode,order_zipcode,Not described in the original dictionary,No,Dropped: about 86% missing,missing from original dictionary
53,(added),order_year,Year the order was made (from order_date),Yes,Added during cleaning,added in cleaning
54,(added),order_month,Month number of the order (1-12),Yes,Added during cleaning,added in cleaning
55,(added),order_month_name,Month name of the order,Yes,Added during cleaning,added in cleaning
56,(added),order_day_name,Day of the week the order was made,Yes,Added during cleaning,added in cleaning
57,(added),shipping_delay_days,"Actual shipping days minus scheduled days (positive = late, negative = early)",Yes,Added during cleaning,added in cleaning


In [6]:
assert desc[["original_field", "column_name", "description"]].notna().all().all()

for c in ["original_field", "column_name", "description", "cleaning_note"]:
    assert (desc[c] == desc[c].str.strip()).all(), f"مسافات زيادة في {c}"
    assert not desc[c].str.contains(r"\s{2,}", regex=True).any(), f"مسافات مزدوجة في {c}"
assert not desc["description"].str.startswith(":").any(), "وصف لسه بيبدأ بـ :"
assert not desc["description"].str.contains(r"\s,", regex=True).any(), "مسافة قبل فاصلة"

assert desc["column_name"].is_unique
assert len(desc) == 58, f"عدد الصفوف {len(desc)}"

assert len(region_list) == 23 and "Northern Europe" in region_list
assert "Northern" not in region_list and "Europe" not in region_list

CLEANED_DATA = OUTPUT_DIR / "DataCo_cleaned.xlsx"
if CLEANED_DATA.exists():
    data_cols = set(pd.read_excel(CLEANED_DATA, nrows=0).columns)
    dict_cols = set(desc.loc[desc["in_cleaned_dataset"] == "Yes", "column_name"])
    assert data_cols == dict_cols, f"فرق: {data_cols ^ dict_cols}"
    print(f"✔ الوصف مطابق لأعمدة {CLEANED_DATA.name} ({len(data_cols)} عمود)")
else:
    print("(ملف DataCo_cleaned.xlsx لسه مش في فولدر files، شغّل DataCo_Cleaning الأول لو عايز المقارنة)")

print("✅ كل الفحوصات عدّت — ملف الوصف نضيف 100%")

(ملف DataCo_cleaned.xlsx لسه مش في فولدر files، شغّل DataCo_Cleaning الأول لو عايز المقارنة)
✅ كل الفحوصات عدّت — ملف الوصف نضيف 100%


In [7]:
print(f"Rows before: 52 → after: {len(desc)}")
print(desc["in_cleaned_dataset"].value_counts().to_dict())

OUTPUT_PATH = OUTPUT_DIR / "DataCo_Description_cleaned.xlsx"
desc.to_excel(OUTPUT_PATH, index=False, sheet_name="description_cleaned", engine=EXCEL_ENGINE)
print("Saved:", OUTPUT_PATH, "→", desc.shape)

desc.head(10)

Rows before: 52 → after: 58
{'Yes': 46, 'No': 12}
Saved: C:\Users\LAP TECH\Desktop\DEPI final Project\files\DataCo_Description_cleaned.xlsx → (58, 6)


,original_field,column_name,description,in_cleaned_dataset,cleaning_note,source
0,Type,payment_type,Type of transaction made,Yes,,original dictionary
1,Days for shipping (real),days_for_shipping_real,Actual shipping days of the purchased product,Yes,,original dictionary
2,Days for shipment (scheduled),days_for_shipment_scheduled,Days of scheduled delivery of the purchased product,Yes,,original dictionary
3,Benefit per order,benefit_per_order,Earnings per order placed,No,Dropped: identical to order_profit_per_order,original dictionary
4,Sales per customer,sales_per_customer,Total sales per customer made per customer,No,Dropped: identical to order_item_total,original dictionary
5,Delivery Status,delivery_status,"Delivery status of orders: Advance shipping, Late delivery, Shipping canceled, Shippin...",Yes,,original dictionary
6,Late_delivery_risk,late_delivery_risk,"Categorical variable that indicates if sending is late (1), it is not late (0).",Yes,,original dictionary
7,Category Id,category_id,Product category code,Yes,,original dictionary
8,Category Name,category_name,Description of the product category,Yes,,original dictionary
9,Customer City,customer_city,City where the customer made the purchase,Yes,,original dictionary
